# Chapters 1.4, 5 and 6: the duty and MRP calculator

The calculator (`calculator.py`) reads a rate table, picks the row in force on a date and returns duty
for designs A, B, B2 and C. Then it runs the walk from declared manufacturing cost (DMC) to MRP.
State S is synthetic. Change one input and rerun to watch the waterfall move.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2
from calculator import load_rates, row_in_force, duty_for, floor_bites_below, walk, NoRate

In [ ]:
# Inputs: change these
DMC = 12600 * 0.078          # INR a case: net price a hL x hL a case
BOTTLES, LITRES, ABV = 12, 0.65, 5.3
DATE = '2026-07-15'
rates = load_rates()
BL = BOTTLES * LITRES
rates[['row_id', 'state', 'kind', 'item', 'band', 'design', 'share', 'floor_per_bl', 'rate_per_bl', 'rate_per_lal', 'fee_amount', 'effective_from', 'effective_to']]

## Duty under the four designs (chapter 5.2)

In [ ]:
designs = {}
for state, label in [('S-A', 'A: 110 percent of DMC'), ('S-B', 'B: INR 140 a bulk litre'),
                     ('S-B2', 'B2: INR 2,650 a litre of alcohol'), ('S', 'C: larger of 110 percent and INR 50 a bulk litre')]:
    row = row_in_force(rates, state, ABV, DATE)
    designs[label] = duty_for(row, DMC, BL, ABV)
    print(f'{label:52s} {inr(designs[label]):>10s}   ({row.row_id})')
floor_case = 50 * BL
floor_line = floor_bites_below(1.10, 50, BL)
print('floor a case', inr(floor_case), '; the floor bites below DMC', inr(floor_line), '=', round(floor_line / BL, 2), 'a bulk litre')

The same beer at three declared costs a bulk litre (the table in 5.2.3).

In [ ]:
per_litre = pd.DataFrame({'DMC a litre': [126.00, 88.20, 40.00]})
per_litre['A: 110 percent'] = (per_litre['DMC a litre'] * 1.10).round(2)
per_litre['B: INR 140'] = 140.00
per_litre['C: larger of A and 50'] = per_litre['A: 110 percent'].clip(lower=50).round(2)
per_litre

## The strength cliff (chapter 5.3)

In [ ]:
strong = duty_for(row_in_force(rates, 'S', 5.3, DATE), DMC, BL, 5.3)
mild = duty_for(row_in_force(rates, 'S', 5.0, DATE), DMC, BL, 5.0)
cliff = r2(strong - mild)
print('strong', inr(strong), '; mild', inr(mild), '; gap', inr(cliff), f'= {cliff / strong:.1%} of the duty;', r2(cliff / 12), 'a bottle')
print(f'alcohol removed {0.3 / 5.3:.1%}; plant duty a hL', inr(strong / 0.078), 'against the brewpub 3,000')
for hl in (500, 50):
    cases = round(hl / 0.078)
    print(f'registration on a {hl} hL SKU: {inr(cases, 0)} cases, INR {25000 / cases:.2f} a case')

## The walk from DMC to MRP (chapters 1.4 and 6.4)

In [ ]:
w = walk(DMC, BOTTLES, LITRES, ABV, DATE, rates)
lines = [('DMC', w.dmc), ('Excise', w.duty), ('Transport pass fee', w.fee), ('Wholesaler margin', w.buyer_margin),
         ('Retailer margin', w.retail_margin), ('Rounding, kept by the retailer', w.rounding)]
table = pd.DataFrame(lines, columns=['step', 'INR a case'])
table['INR a bottle'] = (table['INR a case'] / BOTTLES).round(2)
table['share of MRP'] = (table['INR a case'] / w.mrp_case * 100).round(2)
print('landed', inr(w.landed), '; wholesaler price', inr(w.buyer_price), '; before rounding', inr(w.pre_round), f'({w.pre_round_bottle} a bottle)')
print('MRP', inr(w.mrp_case), 'a case,', w.mrp_bottle, f'a bottle; state share {w.state_share:.1%}; rate version {w.rate_version}')
table

In [ ]:
margin_over_delivered = r2(w.dmc - 667.20)   # delivered cost from chapter 9.6
print('Kestrel margin', inr(margin_over_delivered), f'= {margin_over_delivered / w.mrp_case:.1%} of MRP; trade keeps',
      f'{(w.buyer_margin + w.retail_margin + w.rounding) / w.mrp_case:.1%}')
fig, ax = plt.subplots(figsize=(8, 3))
base = 0
for step, v in zip(table.step, table['INR a bottle']):
    ax.bar(step, v, bottom=base); base += v
ax.set_ylabel('INR a bottle'); ax.set_title(f'DMC to MRP of INR {w.mrp_bottle:.0f}, synthetic State S'); plt.xticks(rotation=30, ha='right'); plt.show()

## No row, no answer
A date with no rate row is a refusal, never a guess.

In [ ]:
try:
    walk(DMC, date='2026-03-15', rates=rates)
except NoRate as e:
    refusal = str(e)
    print('refused:', refusal)